In [0]:
# Step 2: Bronze layer - raw data, with only column names made legal

CATALOG = "superstore"
SCHEMA  = "default"

bronze_path  = f"/Volumes/{CATALOG}/{SCHEMA}/superstore/superstore.csv"
bronze_table = f"{CATALOG}.{SCHEMA}.superstore_bronze"

raw_df = (
    spark.read
        .format("csv")
        .option("header", "true")
        .option("inferSchema", "true")
        .option("encoding", "ISO-8859-1")
        .option("multiLine", "true")
        .option("escape", '"')
        .load(bronze_path)
)

# make every column name legal for Delta: no spaces, no stray punctuation
clean_names = [c.strip().replace(" ", "_") for c in raw_df.columns]
bronze_df = raw_df.toDF(*clean_names)

print("Columns now:", bronze_df.columns)

(
    bronze_df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(bronze_table)
)

print("Bronze rows:", bronze_df.count())
bronze_df.printSchema()

Columns now: ['ID', 'Order_id', 'Order_Date', 'Ship__Date', 'Ship_Mode', 'Customer_id', 'Customer_Name', 'Segment', 'Country', 'City', 'State', 'Postal_Code', 'Region', 'Product__ID', 'Category', 'Sub_Category', 'Product_Name', 'Sales', 'Quantity', 'Discount', 'Profit', 'user_id', 'state_id', 'order_s']
Bronze rows: 9994
root
 |-- ID: integer (nullable = true)
 |-- Order_id: string (nullable = true)
 |-- Order_Date: string (nullable = true)
 |-- Ship__Date: date (nullable = true)
 |-- Ship_Mode: string (nullable = true)
 |-- Customer_id: string (nullable = true)
 |-- Customer_Name: string (nullable = true)
 |-- Segment: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- City: string (nullable = true)
 |-- State: string (nullable = true)
 |-- Postal_Code: integer (nullable = true)
 |-- Region: string (nullable = true)
 |-- Product__ID: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Sub_Category: string (nullable = true)
 |-- Product_Name: string 

In [0]:
# Step 3: Silver layer - clean, typed, business-ready

from pyspark.sql.functions import col, try_to_date, coalesce, year, month, trim

CATALOG = "superstore"
SCHEMA  = "default"

bronze = spark.table(f"{CATALOG}.{SCHEMA}.superstore_bronze")

silver_df = (
    bronze
        # 1. keep only the columns the business needs, and give them clean names
        .select(
            col("ID").alias("row_id"),
            col("Order_id").alias("order_id"),
            col("Order_Date").alias("order_date_raw"),
            col("Ship__Date").alias("ship_date"),
            col("Ship_Mode").alias("ship_mode"),
            col("Customer_id").alias("customer_id"),
            trim(col("Customer_Name")).alias("customer_name"),
            col("Segment").alias("segment"),
            col("Country").alias("country"),
            col("City").alias("city"),
            col("State").alias("state"),
            col("Postal_Code").alias("postal_code"),
            col("Region").alias("region"),
            col("Product__ID").alias("product_id"),
            col("Category").alias("category"),
            col("Sub_Category").alias("sub_category"),
            trim(col("Product_Name")).alias("product_name"),
            col("Sales").cast("double").alias("sales"),
            col("Quantity").cast("int").alias("quantity"),
            col("Discount").cast("double").alias("discount"),
            col("Profit").cast("double").alias("profit"),
        )
        # 2. repair the date: try the normal format, then the odd one
        .withColumn(
            "order_date",
            coalesce(
                try_to_date(col("order_date_raw"), "yyyy-MM-dd"),
                try_to_date(col("order_date_raw"), "dd/MM/yyyy"),
            )
        )
        .drop("order_date_raw")
        # 3. add columns the dashboard will want
        .withColumn("order_year",  year(col("order_date")))
        .withColumn("order_month", month(col("order_date")))
)

print("Silver rows:", silver_df.count())
silver_df.printSchema()

Silver rows: 9994
root
 |-- row_id: integer (nullable = true)
 |-- order_id: string (nullable = true)
 |-- ship_date: date (nullable = true)
 |-- ship_mode: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- segment: string (nullable = true)
 |-- country: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- postal_code: integer (nullable = true)
 |-- region: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- category: string (nullable = true)
 |-- sub_category: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- sales: double (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- discount: double (nullable = true)
 |-- profit: double (nullable = true)
 |-- order_date: date (nullable = true)
 |-- order_year: integer (nullable = true)
 |-- order_month: integer (nullable = true)



In [0]:
# Step 3b: did any dates fail to parse?

bad_dates = silver_df.filter(col("order_date").isNull())
print("Rows with unparseable order_date:", bad_dates.count())
display(bad_dates)

Rows with unparseable order_date: 1


row_id,order_id,ship_date,ship_mode,customer_id,customer_name,segment,country,city,state,postal_code,region,product_id,category,sub_category,product_name,sales,quantity,discount,profit,order_date,order_year,order_month
3758,CA-2023-111283,2023-03-04,Standard Class,LC-16870,Lena Cacioppo,Consumer,United States,Newark,Ohio,43055,East,OFF-AR-10001615,Office Supplies,Art,Newell 34,111.104,7,0.2,8.3328,null,null,null


In [0]:
# Step 3c: split good rows from bad, then write both tables

silver_table     = f"{CATALOG}.{SCHEMA}.superstore_silver"
quarantine_table = f"{CATALOG}.{SCHEMA}.superstore_quarantine"

good_rows = silver_df.filter(col("order_date").isNotNull())
bad_rows  = silver_df.filter(col("order_date").isNull())

(
    good_rows.write
        .format("delta").mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(silver_table)
)

(
    bad_rows.write
        .format("delta").mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(quarantine_table)
)

bronze_count = bronze.count()
good_count   = good_rows.count()
bad_count    = bad_rows.count()

print("Bronze rows     :", bronze_count)
print("Silver rows     :", good_count)
print("Quarantined rows:", bad_count)
print("Reconciles      :", bronze_count == good_count + bad_count)

Bronze rows     : 9994
Silver rows     : 9993
Quarantined rows: 1
Reconciles      : True


In [0]:
# Step 4: the four headline numbers

from pyspark.sql.functions import col, sum, countDistinct

silver = spark.table("superstore.default.superstore_silver")

kpi_df = silver.agg(
    countDistinct("customer_id").alias("total_customers"),
    countDistinct("order_id").alias("total_orders"),
    sum("sales").alias("total_sales"),
    sum("profit").alias("total_profit")
)

kpi_df.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("superstore.default.gold_kpis")

display(kpi_df)

total_customers,total_orders,total_sales,total_profit
793,5008,2297089.7562999553,286388.6917000013


In [0]:
# Q5: total sales by country

sales_by_country = silver.groupBy("country") \
    .agg(sum("sales").alias("total_sales")) \
    .orderBy(col("total_sales").desc())

sales_by_country.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("superstore.default.gold_sales_by_country")

display(sales_by_country)

country,total_sales
United States,2297089.7562999553


In [0]:
'''
top_states = silver.groupBy("state") \
    .agg(sum("sales").alias("total_sales")) \
    .orderBy(col("total_sales").desc()) \
    .limit(10)

top_states.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("superstore.default.gold_top_states")

display(top_states)
'''

state,total_sales
California,457687.631500001
New York,310876.2709999998
Texas,170188.04580000002
Washington,138641.26999999993
Pennsylvania,116511.91400000003
Florida,89473.708
Illinois,80166.10099999986
Ohio,78147.03199999993
Michigan,76269.61400000002
Virginia,70636.71999999999


In [0]:
# Q6: most profitable region

profit_by_region = silver.groupBy("region") \
    .agg(sum("profit").alias("total_profit")) \
    .orderBy(col("total_profit").desc())

profit_by_region.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("superstore.default.gold_profit_by_region")

display(profit_by_region)

region,total_profit
West,108418.45030000017
East,91514.44720000026
South,46749.430300000065
Central,39706.36389999999


In [0]:
# Q7: top selling categories

sales_by_category = silver.groupBy("category") \
    .agg(sum("sales").alias("total_sales")) \
    .orderBy(col("total_sales").desc())

sales_by_category.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("superstore.default.gold_sales_by_category")

display(sales_by_category)

category,total_sales
Technology,836154.0329999966
Furniture,741999.7952999998
Office Supplies,718935.928000003


In [0]:
# Q8: top 10 sub categories by sales

top_sub_categories = silver.groupBy("sub_category") \
    .agg(sum("sales").alias("total_sales")) \
    .orderBy(col("total_sales").desc()) \
    .limit(10)

top_sub_categories.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("superstore.default.gold_top_sub_categories")

display(top_sub_categories)

sub_category,total_sales
Phones,330007.0540000001
Chairs,328449.10300000076
Storage,223843.60800000012
Tables,206965.5320000001
Binders,203412.7330000001
Machines,189238.63099999996
Accessories,167380.3180000001
Copiers,149528.02999999994
Bookcases,114879.99629999997
Appliances,107532.161


In [0]:
# Q9: most ordered product, by quantity not money

most_ordered = silver.groupBy("product_name") \
    .agg(sum("quantity").alias("total_quantity")) \
    .orderBy(col("total_quantity").desc()) \
    .limit(10)

most_ordered.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("superstore.default.gold_most_ordered_products")

display(most_ordered)

product_name,total_quantity
Staples,215
Staple envelope,170
Easy-staple paper,150
Staples in misc. colors,86
KI Adjustable-Height Table,74
Avery Non-Stick Binders,71
Storex Dura Pro Binders,71
GBC Premium Transparent Covers with Diagonal Lined Pattern,67
"Situations Contoured Folding Chairs, 4/Set",64
Staple-based wall hangings,62


In [0]:
# Q10: top 10 customers by sales

top_customers = silver.groupBy("customer_name") \
    .agg(sum("sales").alias("total_sales")) \
    .orderBy(col("total_sales").desc()) \
    .limit(10)

top_customers.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("superstore.default.gold_top_customers")

display(top_customers)

customer_name,total_sales
Sean Miller,25043.05
Tamara Chand,19052.217999999997
Raymond Buch,15117.339
Tom Ashbrook,14595.62
Adrian Barton,14473.570999999998
Ken Lonsdale,14175.229
Sanjit Chand,14142.333999999999
Hunter Lopez,12873.297999999999
Sanjit Engle,12209.438000000002
Christopher Conant,12129.072


In [0]:
# Q10b: top 10 cities by sales

top_cities = silver.groupBy("city") \
    .agg(sum("sales").alias("total_sales")) \
    .orderBy(col("total_sales").desc()) \
    .limit(10)

top_cities.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("superstore.default.gold_top_cities")

display(top_cities)

city,total_sales
New York City,256368.161
Los Angeles,175851.341
Seattle,119540.742
San Francisco,112669.09199999992
Philadelphia,109077.01300000008
Houston,64504.76039999994
Chicago,48539.541000000034
San Diego,47521.028999999995
Jacksonville,44713.183
Springfield,43054.342000000004
